In [ ]:
!pip install -r requirements.txt
from warpdrive import WarpDrive
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from catboost import CatBoostClassifier
import pickle
import pandas as pd

wd = WarpDrive()

# Fetch argument names from WarpDrive
col1 = wd.get_args("col1")
dataset1 = wd.get_args("dataset1")
model1 = wd.get_args("model1")
console1 = wd.get_args("console1")
string1 = wd.get_args("string1")
int1 = wd.get_args("int1")
float1 = wd.get_args("float1")
bool1 = wd.get_args("bool1")

# Save dataset and visuals
wd.save_table(dataset1)
wd.create_df(dataset1)
wd.save_png(console1)

# Logistic Regression
X = dataset1[['Loan_Amount', 'Home_Owner']]
y = dataset1['Gender']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

exog_columns = ["Loan_Amount", "Home_Owner"]
wd.create_model(
    model=model,
    library="sklearn",
    model_technique="LogisticRegressionClassifier",
    input_variables=exog_columns,
    target_column="Gender",
    train_table="dataset1",
    lags=0,
    exog_columns=exog_columns,
)
with open('logistic_model.pkl', 'wb') as file:
    pickle.dump(model, file)

# CatBoost Classifier
clf = CatBoostClassifier(random_state=0).fit(dataset1[exog_columns].values, dataset1['Gender'].values)
wd.create_model(
    model=clf,
    library="catboost",
    model_technique="CatboostClassifier",
    input_variables=exog_columns,
    target_column="Gender",
    train_table="dataset1",
    lags=0,
    exog_columns=exog_columns,
)
with open('cbc_model.pkl', 'wb') as file:
    pickle.dump(clf, file)
    
dataset1['Date'] = pd.to_datetime(dataset1['Date'])
dataset1.set_index('Date', inplace=True)
modelts = SARIMAX(y, exog=X, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0))
results = modelts.fit(disp=False)
print(results.summary())   
wd.create_model(
    model=modelts,
    library="statsmodels",
    model_technique="ARIMA",
    input_variables=exog_columns,  # Only exogenous columns are used
    time_column="Date",
    target_column="Gender",
    train_table="df", 
    lags=0,
    exog_columns=exog_columns,
)
with open('arima_model.pkl', 'wb') as file:
    pickle.dump(results, file)
# Update dataset
#dataset1.index = range(len(dataset1))
#dataset1['Age_copy'] = dataset1['Age']
#wd.update_df(dataset1, "dataset1")
#wd.create_df(dataset1, "UDF_UPDATED_DF")

# Map categorical values
dataset1['Gender'] = dataset1['Gender'].map({0: 'Female', 1: 'Male'})
dataset1['Home_Owner'] = dataset1['Home_Owner'].map({0: 'Renter', 1: 'Owner'})

# Plots
fig1 = px.box(dataset1, x="Gender", y="Income", color="Home_Owner", points="all")
wd.save_graph(fig1)

fig2 = px.scatter(dataset1, x="Income", y="Credit_Score", color="Gender", hover_data=["Home_Owner"])
wd.save_graph(fig2)

plt.figure(figsize=(6, 4))
sns.heatmap(dataset1[['Income', 'Credit_Score']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
wd.save_image(plt)

fig3 = px.histogram(dataset1, x="Home_Owner", color="Gender", barmode="group")
wd.save_graph(fig3)

fig4 = px.scatter(dataset1, x="Income", y="Credit_Score", color="Home_Owner", hover_data=["Gender"])
wd.save_graph(fig4)

plt.figure(figsize=(6, 4))
sns.kdeplot(data=dataset1, x="Income", hue="Gender", fill=True)
plt.title("Income Distribution by Gender (KDE Plot)")
plt.tight_layout()
wd.save_image(plt)

# Output artifacts
wd.add_output_artifact("INTEGER", int1)
wd.add_output_artifact("FLOAT", float1)
wd.add_output_artifact("BOOLEAN", bool1)
wd.add_output_artifact("STRING", string1)
wd.save_console_file("console1")
